In [1]:
def clean_conll_file(input_file_path: str, output_file_path: str):
    """
    Reads a CoNLL file and corrects inconsistencies in parenthesis tagging.
    It enforces the rule that parentheses surrounding an entity should be
    part of the entity span.
    """
    cleaned_lines = []
    with open(input_file_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    # Process each sentence block
    current_sentence = []
    for line in lines:
        line = line.strip()
        if line:
            current_sentence.append(line.split('\t'))
        else:
            if current_sentence:
                # Process the sentence for corrections
                new_sentence = []
                i = 0
                while i < len(current_sentence):
                    token, tag = current_sentence[i]

                    # Pattern 1: Merging a leading '('
                    if token == '(' and tag == 'O' and i + 1 < len(current_sentence) and current_sentence[i+1][1] != 'O':
                        # The next token is part of an entity. Merge the parenthesis.
                        new_tag = current_sentence[i+1][1]
                        new_tag_prefix = new_tag.split('-')[0]
                        new_tag_entity = new_tag.split('-')[1]
                        if new_tag_prefix in ['B', 'S']:
                            new_sentence.append([token, f'B-{new_tag_entity}'])
                        else:
                             # This case should not happen with proper B-I-E-S tags
                            new_sentence.append([token, f'S-{new_tag_entity}'])
                    
                    # Pattern 2: Merging a trailing ')'
                    elif token == ')' and tag == 'O' and i > 0 and new_sentence[-1][1] != 'O':
                        # The previous token was part of an entity. Merge the parenthesis.
                        prev_tag = new_sentence[-1][1]
                        prev_tag_prefix = prev_tag.split('-')[0]
                        prev_tag_entity = prev_tag.split('-')[1]
                        if prev_tag_prefix in ['E', 'I']:
                            new_sentence.append([token, f'E-{prev_tag_entity}'])
                        else:
                            # This should cover single-token entities
                            new_sentence.append([token, f'S-{prev_tag_entity}'])
                            new_sentence[-2][1] = 'B-'+prev_tag_entity
                    
                    # Pattern 3: Regular token, or parenthesis not matching the patterns
                    else:
                        new_sentence.append([token, tag])
                    i += 1
                
                # Write the corrected sentence to our list
                for corrected_token, corrected_tag in new_sentence:
                    cleaned_lines.append(f"{corrected_token}\t{corrected_tag}\n")
                cleaned_lines.append("\n") # Add blank line

            current_sentence = []

    # Write the cleaned data to the output file
    with open(output_file_path, 'w', encoding='utf-8') as f:
        f.writelines(cleaned_lines)

# --- Usage ---
input_conll_file = 'annotated_data.conll'
output_conll_file = 'cleaned_annotated_data.conll'

clean_conll_file(input_conll_file, output_conll_file)
print(f"Annotation cleanup complete. Cleaned data saved to {output_conll_file}")

Annotation cleanup complete. Cleaned data saved to cleaned_annotated_data.conll
